# ACS Poststratification Frames

Loads pre-recoded adults from `data/acs_adults_model_vars.parquet` (run `extract_acs.ipynb` first) and produces:

- **`data/poststrat_state.csv`** — `state_fips × gender × race4 × educ_category → N`
- **`data/poststrat_county.csv`** — `county_fips × gender × race4 × educ_category → N` + county covariates

**Prerequisites:** run `extract_acs.ipynb` once to generate the parquet cache.

In [1]:
import pandas as pd
import numpy as np
import os, urllib.request

DATA_DIR = "/Users/carmenk/Documents/CSS/Capstone/data/"

pums = pd.read_parquet(DATA_DIR + "acs_adults_model_vars.parquet")
print(f"Loaded {len(pums):,} adults | columns: {pums.columns.tolist()}")
print(f"Weighted adult population: {pums['PWGTP'].sum():,.0f}\n")

# Recoding applied in extract_acs.ipynb:
#   gender       : SEX → Male / Female
#   race4        : HISP (precedence) + RAC1P → White / Black / Hispanic / Other
#   educ_category: SCHL bins [0,15,17,20,24] → 1 LessHS / 2 HS / 3 SomeCol / 4 BA+
educ_labels = {1: "Less than HS", 2: "High school", 3: "Some college", 4: "Bachelor's or higher"}
division_map = {
    1: "New England",      2: "Mid-Atlantic",       3: "E. North Central",
    4: "W. North Central", 5: "South Atlantic",     6: "E. South Central",
    7: "W. South Central", 8: "Mountain",           9: "Pacific",
}

print("gender:\n", pums["gender"].value_counts().to_string())
print("\nrace4:\n", pums["race4"].value_counts().to_string())
print("\neduc_category:")
print(pums["educ_category"].map(educ_labels).value_counts()
      .reindex(educ_labels.values()).to_string())

Loaded 13,044,338 adults | columns: ['DIVISION', 'STATE', 'PWGTP', 'gender', 'race4', 'educ_category', 'region9', 'state_fips', 'puma_code']
Weighted adult population: 261,411,680

gender:
 gender
Female    6699173
Male      6345165

race4:
 race4
White       8642908
Hispanic    1807760
Other       1449945
Black       1143725

educ_category:
educ_category
Less than HS            1296394
High school             3565679
Some college            3806362
Bachelor's or higher    4375903


## 1. State-Level Poststratification Frame

Group by `state_fips × gender × race4 × educ_category`, summing `PWGTP`.  
Result: up to 51 × 2 × 4 × 4 = 1,632 cells.

In [2]:
valid = pums.dropna(subset=["gender", "race4", "educ_category"])

poststrat_state = (
    valid
    .groupby(["state_fips", "DIVISION", "gender", "race4", "educ_category"], observed=True)["PWGTP"]
    .sum()
    .reset_index()
    .rename(columns={"PWGTP": "N", "DIVISION": "division"})
)
poststrat_state["region9"]    = poststrat_state["division"].map(division_map)
poststrat_state["educ_label"] = poststrat_state["educ_category"].map(educ_labels)

print(f"State frame: {len(poststrat_state):,} rows | {poststrat_state['state_fips'].nunique()} states")
print(f"Total weighted adult pop: {poststrat_state['N'].sum():>15,.0f}")

cell_counts = poststrat_state.groupby("state_fips").size()
incomplete = cell_counts[cell_counts < 32]
if incomplete.empty:
    print("All 51 states have all 32 demographic cells ✓")
else:
    print(f"Sparse states: {incomplete.to_dict()}")

OUT = DATA_DIR + "poststrat_state.csv"
poststrat_state.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")
print(f"Columns: {poststrat_state.columns.tolist()}")

State frame: 1,632 rows | 51 states
Total weighted adult pop:     261,411,680
All 51 states have all 32 demographic cells ✓

Saved → /Users/carmenk/Documents/CSS/Capstone/data/poststrat_state.csv
Columns: ['state_fips', 'division', 'gender', 'race4', 'educ_category', 'N', 'region9', 'educ_label']


## 2. County-Level Poststratification Frame

ACS PUMS identifies geography as `STATE + PUMA` — there is no county field.  
A **Census Tract → PUMA crosswalk** allocates each PUMA's population across the counties it intersects,
proportional to the number of census tracts in each PUMA-county pair.

County-level covariates added: `co2_per_capita` and `dem_share_two_party`.

In [3]:
# ── County-level covariates ───────────────────────────────────────────────────
carbon = pd.read_csv(DATA_DIR + "carbon_county.csv", dtype={"county_fips": str, "state_fips": str})
pres   = pd.read_csv(DATA_DIR + "pres_county.csv",   dtype={"county_fips": str, "state_fips": str})
carbon["county_fips"] = carbon["county_fips"].str.zfill(5)
pres["county_fips"]   = pres["county_fips"].str.zfill(5)

covars = (
    carbon[["county_fips", "state_fips", "county_name", "state_abbr", "co2_per_capita"]]
    .merge(pres[["county_fips", "dem_share_two_party"]], on="county_fips", how="outer")
)
print(f"Covariates: {len(covars):,} counties | "
      f"missing co2: {covars['co2_per_capita'].isna().sum()} | "
      f"missing dem_share: {covars['dem_share_two_party'].isna().sum()}")

# ── PUMA-to-county crosswalk (Census Tract → PUMA relationship file) ──────────
TRACT_PUMA_PATH = DATA_DIR + "tract_to_puma_2020.txt"
TRACT_PUMA_URL  = ("https://www2.census.gov/geo/docs/maps-data/data/rel2020/"
                   "2020_Census_Tract_to_2020_PUMA.txt")

if not os.path.exists(TRACT_PUMA_PATH):
    urllib.request.urlretrieve(TRACT_PUMA_URL, TRACT_PUMA_PATH)
    print(f"Downloaded → {TRACT_PUMA_PATH}")

tracts = pd.read_csv(TRACT_PUMA_PATH, dtype=str)
tracts.columns = [c.replace("20", "") for c in tracts.columns]
tracts["state_fips"]  = tracts["STATEFP"]
tracts["county_fips"] = tracts["STATEFP"] + tracts["COUNTYFP"]
tracts["puma_code"]   = tracts["PUMA5CE"].str.zfill(5)

puma_county = (
    tracts.groupby(["state_fips", "puma_code", "county_fips"])
    .size().reset_index(name="n_tracts")
)
puma_county["afact"] = (
    puma_county["n_tracts"] /
    puma_county.groupby(["state_fips", "puma_code"])["n_tracts"].transform("sum")
)
xwalk = puma_county[["state_fips", "puma_code", "county_fips", "afact"]]
print(f"Crosswalk: {len(xwalk):,} PUMA-county pairs | {xwalk['county_fips'].nunique():,} counties")

Covariates: 3,191 counties | missing co2: 47 | missing dem_share: 46
Crosswalk: 4,701 PUMA-county pairs | 3,222 counties


In [4]:
valid_county = pums.dropna(subset=["gender", "race4", "educ_category"])

# Step 1: sum weights within each PUMA × demographic cell
puma_cells = (
    valid_county
    .groupby(["state_fips", "puma_code", "DIVISION", "gender", "race4", "educ_category"], observed=True)["PWGTP"]
    .sum()
    .reset_index(name="N_puma")
)
print(f"PUMA-level cells: {len(puma_cells):,}")

# Step 2: distribute weight across counties by allocation factor
allocated = puma_cells.merge(xwalk, on=["state_fips", "puma_code"], how="left")
n_unmapped = allocated["county_fips"].isna().sum()
if n_unmapped:
    print(f"PUMA cells without county mapping (dropped): {n_unmapped:,}")
    allocated = allocated.dropna(subset=["county_fips"])
allocated["N"] = allocated["N_puma"] * allocated["afact"]

# Step 3: aggregate to county × demographic
poststrat_county = (
    allocated
    .groupby(["county_fips", "state_fips", "DIVISION", "gender", "race4", "educ_category"], observed=True)["N"]
    .sum().reset_index()
)
poststrat_county["region9"]    = poststrat_county["DIVISION"].map(division_map)
poststrat_county["educ_label"] = poststrat_county["educ_category"].map(educ_labels)

# Step 4: merge county covariates
poststrat_county = poststrat_county.merge(
    covars[["county_fips", "county_name", "state_abbr", "co2_per_capita", "dem_share_two_party"]],
    on="county_fips", how="left"
)

print(f"\nCounty frame: {len(poststrat_county):,} rows | {poststrat_county['county_fips'].nunique():,} counties")
print(f"Total weighted adult pop: {poststrat_county['N'].sum():>15,.0f}")
print(f"Missing co2: {poststrat_county['co2_per_capita'].isna().sum():,} | "
      f"missing dem_share: {poststrat_county['dem_share_two_party'].isna().sum():,}")

OUT = DATA_DIR + "poststrat_county.csv"
poststrat_county.to_csv(OUT, index=False)
print(f"\nSaved → {OUT}")
print(f"Columns: {poststrat_county.columns.tolist()}")

PUMA-level cells: 78,569

County frame: 99,940 rows | 3,143 counties
Total weighted adult pop:     261,411,680
Missing co2: 256 | missing dem_share: 1,167

Saved → /Users/carmenk/Documents/CSS/Capstone/data/poststrat_county.csv
Columns: ['county_fips', 'state_fips', 'DIVISION', 'gender', 'race4', 'educ_category', 'N', 'region9', 'educ_label', 'county_name', 'state_abbr', 'co2_per_capita', 'dem_share_two_party']
